In [ ]:
from __future__ import annotations

from pathlib import Path
import gc
import re
import traceback
import warnings

import numpy as np
import pandas as pd
import xarray as xr

from scipy import special
from scipy.spatial import cKDTree

import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.patches import Patch


# =============================================================================
# 0. USER SETTINGS
# =============================================================================

ROOT = Path(r"F:\CMIP6_STORAGE_STATE")

MANIFEST_EXACT = ROOT / "CMIP6_download_MODELS.csv"

OUTPUT_ROOT = (
    ROOT
    / "DIRECT_STORAGE_FUTURE_ANALYSIS"
)

WORLD_SHP = Path(
    r"D:\World_Countries_Generalized.shp"
)

# Periods.
EARLY_START = 1979
EARLY_END = 2000

RECENT_START = 2001
RECENT_END = 2023

HIST_RECENT_END = 2014
SCEN_RECENT_START = 2015

FUTURE_WINDOWS = {
    "ssp245": {
        "mid_2041_2060": (2041, 2060),
        "late_2081_2100": (2081, 2100),
    },
    "ssp585": {
        "mid_2041_2060": (2041, 2060),
        "late_2081_2100": (2081, 2100),
    },
}


HIST_MANIFEST_PERIOD = "historical_1979_2014"

# Common grid.
TARGET_RESOLUTION_DEG = 1.0

# Trend / significance settings.
SLOPE_EPS = 1.0e-6
P_THRESHOLD = 0.05

MIN_EARLY_TREND_YEARS = 18
MIN_RECENT_TREND_YEARS = 19
MIN_FUTURE_TREND_YEARS = 15

MIN_REFERENCE_YEARS = 18
MIN_STD = 1.0e-8

# Multi-model robustness.
ROBUST_AGREEMENT = 2.0 / 3.0
MIN_MODELS_PER_CELL = 15

MIN_MONTHS_PER_YEAR = 12

# Execution.
OVERWRITE = False
MAKE_FIGURES = False
STRICT_UNITS = True
REQUIRE_COUNTRY_LAND_MASK = True

OUTPUT_FLOAT_DTYPE = "float32"

DIR_MODEL = OUTPUT_ROOT / "01_model_period_diagnostics"
DIR_ENSEMBLE = OUTPUT_ROOT / "02_multimodel_ensemble"
DIR_TABLES = OUTPUT_ROOT / "03_tables"
DIR_FIGURES = OUTPUT_ROOT / "04_figures"
DIR_LOGS = OUTPUT_ROOT / "05_logs"

for _d in [
    OUTPUT_ROOT,
    DIR_MODEL,
    DIR_ENSEMBLE,
    DIR_TABLES,
    DIR_FIGURES,
    DIR_LOGS,
]:
    _d.mkdir(parents=True, exist_ok=True)


try:
    import dask.array as da_dask  # noqa: F401
    HAS_DASK = True
except Exception:
    HAS_DASK = False

try:
    import netCDF4  # noqa: F401
    HAS_NETCDF4 = True
except Exception:
    HAS_NETCDF4 = False

try:
    import geopandas as gpd
    HAS_GEOPANDAS = True
except Exception:
    HAS_GEOPANDAS = False

try:
    from rasterio.features import geometry_mask
    from rasterio.transform import from_origin
    HAS_RASTERIO = True
except Exception:
    HAS_RASTERIO = False


def make_target_centers(resolution: float):
    nlat = int(round(180.0 / resolution))
    nlon = int(round(360.0 / resolution))

    lat = -90.0 + resolution / 2.0 + np.arange(nlat) * resolution
    lon = -180.0 + resolution / 2.0 + np.arange(nlon) * resolution

    return lat.astype(np.float64), lon.astype(np.float64)


TARGET_LAT, TARGET_LON = make_target_centers(
    TARGET_RESOLUTION_DEG
)



def build_country_land_mask():
    if not WORLD_SHP.exists():
        message = f"World shapefile not found: {WORLD_SHP}"

        if REQUIRE_COUNTRY_LAND_MASK:
            raise FileNotFoundError(message)

        warnings.warn(
            message + " Falling back to model-valid cells."
        )

        return xr.DataArray(
            np.ones(
                (len(TARGET_LAT), len(TARGET_LON)),
                dtype=bool,
            ),
            dims=("lat", "lon"),
            coords={
                "lat": TARGET_LAT,
                "lon": TARGET_LON,
            },
            name="country_land_mask",
        )

    if not HAS_GEOPANDAS or not HAS_RASTERIO:
        message = (
            "geopandas and rasterio are required for the "
            "global-country land mask."
        )

        if REQUIRE_COUNTRY_LAND_MASK:
            raise ImportError(message)

        warnings.warn(
            message + " Falling back to model-valid cells."
        )

        return xr.DataArray(
            np.ones(
                (len(TARGET_LAT), len(TARGET_LON)),
                dtype=bool,
            ),
            dims=("lat", "lon"),
            coords={
                "lat": TARGET_LAT,
                "lon": TARGET_LON,
            },
            name="country_land_mask",
        )

    world = gpd.read_file(WORLD_SHP)

    if world.crs is None:
        raise ValueError(
            f"World shapefile has no CRS: {WORLD_SHP}"
        )

    world = world.to_crs("EPSG:4326")

    geometries = [
        geom
        for geom in world.geometry
        if geom is not None and not geom.is_empty
    ]

    if not geometries:
        raise ValueError(
            f"No valid geometries in {WORLD_SHP}"
        )

    res = TARGET_RESOLUTION_DEG

    transform = from_origin(
        -180.0,
        90.0,
        res,
        res,
    )

    mask_north_to_south = geometry_mask(
        geometries,
        out_shape=(
            len(TARGET_LAT),
            len(TARGET_LON),
        ),
        transform=transform,
        invert=True,
        all_touched=False,
    )

    # TARGET_LAT is south -> north.
    mask = np.flipud(
        mask_north_to_south
    )

    out = xr.DataArray(
        mask.astype(bool),
        dims=("lat", "lon"),
        coords={
            "lat": TARGET_LAT,
            "lon": TARGET_LON,
        },
        name="country_land_mask",
    )

    out.attrs.update(
        {
            "source": str(WORLD_SHP),
            "crs": "EPSG:4326",
            "definition": (
                "1-degree cell center lies within rasterized "
                "global-country polygons"
            ),
        }
    )

    return out


COUNTRY_LAND_MASK = build_country_land_mask()



CLASS_NAMES = {
    0: "Unclassified",
    1: "WW",
    2: "DD",
    3: "WD",
    4: "DW",
}

CLASS_LONG_NAMES = {
    1: "wetting-to-wetting",
    2: "drying-to-drying",
    3: "wetting-to-drying",
    4: "drying-to-wetting",
}

CLASS_COLORS = [
    "#5B7FA3",
    "#A65A5A",
    "#C49A55",
    "#6E9277",
]


def print_header(text: str):
    print("\n" + "=" * 100)
    print(text)
    print("=" * 100)


def safe_name(text: str) -> str:
    return (
        str(text)
        .replace("/", "_")
        .replace("\\", "_")
        .replace(" ", "_")
        .replace(":", "_")
    )


def find_complete_manifest():
    if MANIFEST_EXACT.exists():
        return MANIFEST_EXACT

    candidates = sorted(
        ROOT.glob(
            "CMIP6_download_COMPLETE_MODELS*.csv"
        )
    )

    if not candidates:
        candidates = sorted(
            ROOT.rglob(
                "CMIP6_download_COMPLETE_MODELS*.csv"
            )
        )

    if not candidates:
        raise FileNotFoundError(
            "Could not find "
            "CMIP6_download_COMPLETE_MODELS*.csv "
            f"under {ROOT}"
        )

    return max(
        candidates,
        key=lambda p: (
            p.stat().st_size,
            p.stat().st_mtime,
        ),
    )


def normalize_unit_string(units) -> str:
    if units is None:
        return ""

    u = str(units).strip().lower()

    for old, new in [
        ("**", "^"),
        ("−", "-"),
        ("–", "-"),
        ("_", " "),
    ]:
        u = u.replace(old, new)

    return " ".join(
        u.split()
    )


def convert_water_storage_to_mm(
    da: xr.DataArray,
    variable: str,
):
    units = normalize_unit_string(
        da.attrs.get("units", "")
    )

    kgm2 = {
        "kg m-2",
        "kg m^-2",
        "kg/m2",
        "kg m^(-2)",
        "kg m-2 water",
    }

    mm = {
        "mm",
        "millimeter",
        "millimeters",
        "mm water equivalent",
    }

    m = {
        "m",
        "meter",
        "metre",
        "meters",
        "metres",
    }

    if units in kgm2:
        out = da

    elif units in mm:
        out = da

    elif units in m:
        out = da * 1000.0

    else:
        message = (
            f"Unknown units for {variable}: {units!r}. "
            "Expected kg m-2, mm, or m water equivalent."
        )

        if STRICT_UNITS:
            raise ValueError(message)

        warnings.warn(
            message + " Values retained unchanged."
        )
        out = da

    out = out.astype("float32")
    out.attrs = dict(da.attrs)
    out.attrs["units"] = "mm"

    # Negative storage values are invalid.
    out = out.where(
        out >= 0.0
    )

    return out


def write_netcdf(
    ds: xr.Dataset,
    path: Path,
):
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    if path.exists() and OVERWRITE:
        path.unlink()

    encoding = {}

    for name, var in ds.data_vars.items():

        if np.issubdtype(
            var.dtype,
            np.floating,
        ):
            enc = {
                "dtype": OUTPUT_FLOAT_DTYPE,
            }

            if HAS_NETCDF4:
                enc.update(
                    {
                        "zlib": True,
                        "complevel": 4,
                    }
                )

            encoding[name] = enc

        elif np.issubdtype(
            var.dtype,
            np.integer,
        ):
            enc = {}

            if HAS_NETCDF4:
                enc.update(
                    {
                        "zlib": True,
                        "complevel": 4,
                    }
                )

            encoding[name] = enc

    kwargs = {
        "encoding": encoding,
    }

    if HAS_NETCDF4:
        kwargs["engine"] = "netcdf4"

    ds.to_netcdf(
        path,
        **kwargs,
    )


MANIFEST = find_complete_manifest()

print_header(
    "DDWW CMIP6 ANALYSIS"
)

print(f"Manifest : {MANIFEST}")
print(f"Output   : {OUTPUT_ROOT}")
print(f"Dask     : {HAS_DASK}")
print(f"netCDF4  : {HAS_NETCDF4}")
print(f"geopandas: {HAS_GEOPANDAS}")
print(f"rasterio : {HAS_RASTERIO}")

manifest = pd.read_csv(
    MANIFEST
)

required_columns = {
    "source_id",
    "member_id",
    "experiment_id",
    "variable_id",
    "period",
    "local_path",
    "status",
}

missing_columns = (
    required_columns
    -
    set(manifest.columns)
)

if missing_columns:
    raise KeyError(
        "Manifest missing columns: "
        f"{sorted(missing_columns)}"
    )

manifest = manifest.loc[
    manifest["status"]
    .astype(str)
    .str.lower()
    .eq("success")
].copy()

direct_manifest = manifest.loc[
    manifest["variable_id"]
    .isin(["mrso", "snw"])
].copy()


# One model = one member.
if "member_rank" in manifest.columns:
    selected_pairs = (
        manifest[
            [
                "source_id",
                "member_id",
                "member_rank",
            ]
        ]
        .drop_duplicates()
        .sort_values(
            [
                "source_id",
                "member_rank",
                "member_id",
            ]
        )
        .groupby(
            "source_id",
            as_index=False,
        )
        .first()
        [
            [
                "source_id",
                "member_id",
            ]
        ]
    )

else:
    selected_pairs = (
        direct_manifest[
            [
                "source_id",
                "member_id",
            ]
        ]
        .drop_duplicates()
        .sort_values(
            [
                "source_id",
                "member_id",
            ]
        )
        .groupby(
            "source_id",
            as_index=False,
        )
        .first()
    )


selected_pairs.to_csv(
    DIR_TABLES
    / "models_selected.csv",
    index=False,
)

print(
    f"\nModels selected from complete manifest: "
    f"{len(selected_pairs)}"
)


_VERSION_RE = re.compile(
    r"^v(\d{8})$"
)


def path_version_number(
    path: Path,
):
    for part in reversed(
        path.parts
    ):
        match = _VERSION_RE.match(
            part
        )

        if match:
            return int(
                match.group(1)
            )

    return -1


_TIME_RANGE_RE = re.compile(
    r"_([0-9]{6,8})-([0-9]{6,8})\.nc$"
)


def file_month_range(path: Path):

    match = _TIME_RANGE_RE.search(
        path.name
    )

    if not match:
        return None

    start_token = match.group(1)
    end_token = match.group(2)

    start_yyyymm = int(
        start_token[:6]
    )

    end_yyyymm = int(
        end_token[:6]
    )

    return (
        start_yyyymm,
        end_yyyymm,
    )


def yyyymm_to_index(value: int):
    year = value // 100
    month = value % 100
    return year * 12 + (month - 1)


def requested_month_coverage(
    paths,
    start_year,
    end_year,
):

    request_start = start_year * 100 + 1
    request_end = end_year * 100 + 12

    req_i0 = yyyymm_to_index(
        request_start
    )

    req_i1 = yyyymm_to_index(
        request_end
    )

    covered = set()

    parsed_any = False

    for path in paths:
        parsed = file_month_range(
            path
        )

        if parsed is None:
            continue

        parsed_any = True

        file_start, file_end = parsed

        i0 = max(
            req_i0,
            yyyymm_to_index(
                file_start
            ),
        )

        i1 = min(
            req_i1,
            yyyymm_to_index(
                file_end
            ),
        )

        if i1 >= i0:
            covered.update(
                range(
                    i0,
                    i1 + 1,
                )
            )

    if not parsed_any:
        return -1

    return len(
        covered
    )


def choose_one_version_group(
    paths,
    start_year,
    end_year,
):

    if not paths:
        return []

    groups = {}

    for path in paths:
        version = path_version_number(
            path
        )

        groups.setdefault(
            version,
            [],
        ).append(
            path
        )

    ranked = []

    for version, group in groups.items():

        coverage = requested_month_coverage(
            group,
            start_year,
            end_year,
        )

        total_bytes = 0

        for path in group:
            try:
                total_bytes += path.stat().st_size
            except OSError:
                pass

        ranked.append(
            (
                coverage,
                total_bytes,
                version,
                len(group),
                group,
            )
        )

    ranked.sort(
        key=lambda item: (
            item[0],
            item[1],
            item[2],
            item[3],
        ),
        reverse=True,
    )

    best_coverage = ranked[0][0]

    expected_months = (
        end_year
        -
        start_year
        +
        1
    ) * 12

    if (
        best_coverage >= 0
        and
        best_coverage < expected_months
    ):
        warnings.warn(
            "Best local ESGF version appears to cover only "
            f"{best_coverage}/{expected_months} requested months "
            f"for {start_year}-{end_year}. The subsequent temporal "
            "coverage check."
        )

    return sorted(
        ranked[0][4],
        key=lambda path: str(path).lower(),
    )


def valid_nc_file(
    p: Path,
):
    try:
        return (
            p.is_file()
            and
            p.suffix.lower() == ".nc"
            and
            p.stat().st_size > 0
        )
    except OSError:
        return False


def manifest_grid_label(
    source_id: str,
    member_id: str,
    experiment_id: str,
    variable_id: str,
):
    g = direct_manifest.loc[
        (
            direct_manifest["source_id"]
            ==
            source_id
        )
        &
        (
            direct_manifest["member_id"]
            ==
            member_id
        )
        &
        (
            direct_manifest["experiment_id"]
            ==
            experiment_id
        )
        &
        (
            direct_manifest["variable_id"]
            ==
            variable_id
        )
    ]

    if (
        not g.empty
        and
        "grid_label" in g.columns
    ):
        values = (
            g["grid_label"]
            .dropna()
            .astype(str)
            .unique()
        )

        if len(values):
            return values[0]

    return None


def discover_existing_files(
    source_id: str,
    member_id: str,
    experiment_id: str,
    variable_id: str,
    start_year: int,
    end_year: int,
):
    """
    Search the already downloaded CMIP6 cache.

    No network access.
    """

    pattern = (
        f"{variable_id}_*_{source_id}_"
        f"{experiment_id}_{member_id}_*.nc"
    )

    candidates = []

    for p in ROOT.rglob(
        pattern
    ):
        if valid_nc_file(
            p
        ):
            candidates.append(
                p
            )

    if not candidates:
        return []

    # Respect the selected grid if available.
    selected_grid = manifest_grid_label(
        source_id,
        member_id,
        experiment_id,
        variable_id,
    )

    if selected_grid:
        filtered = [
            p
            for p in candidates
            if f"_{selected_grid}_" in p.name
        ]

        if filtered:
            candidates = filtered

    return choose_one_version_group(
        candidates,
        start_year,
        end_year,
    )


LAT_NAMES = [
    "lat",
    "latitude",
    "nav_lat",
    "rlat",
    "y",
]

LON_NAMES = [
    "lon",
    "longitude",
    "nav_lon",
    "rlon",
    "x",
]


def preprocess_keep_variable(
    ds,
    variable,
):
    if variable not in ds:
        raise KeyError(
            f"{variable} missing from file. "
            f"Available={list(ds.data_vars)}"
        )

    return ds[
        [variable]
    ]


def open_many(
    paths,
    variable,
):
    paths = [
        str(p)
        for p in paths
    ]

    if HAS_DASK:
        try:
            return xr.open_mfdataset(
                paths,
                combine="by_coords",
                preprocess=lambda ds: (
                    preprocess_keep_variable(
                        ds,
                        variable,
                    )
                ),
                data_vars="minimal",
                coords="minimal",
                compat="override",
                combine_attrs="override",
                chunks={
                    "time": 12,
                },
                parallel=False,
                decode_times=True,
            )

        except Exception as exc:
            print(
                f"open_mfdataset failed for {variable}; "
                f"using manual concat.\nReason: {exc}"
            )

    opened = []

    for p in paths:
        ds = xr.open_dataset(
            p,
            decode_times=True,
        )

        if variable not in ds:
            ds.close()

            raise KeyError(
                f"{variable} missing from {p}"
            )

        opened.append(
            ds[
                [variable]
            ]
        )

    if len(opened) == 1:
        return opened[0]

    return xr.concat(
        opened,
        dim="time",
        data_vars="minimal",
        coords="minimal",
        compat="override",
        combine_attrs="override",
    )


def find_coordinate_name(
    ds,
    kind,
):
    preferred = (
        LAT_NAMES
        if kind == "lat"
        else LON_NAMES
    )

    standard_name = (
        "latitude"
        if kind == "lat"
        else "longitude"
    )

    for name in preferred:
        if (
            name in ds.coords
            or
            name in ds.variables
        ):
            return name

    for name, var in ds.variables.items():

        if (
            str(
                var.attrs.get(
                    "standard_name",
                    "",
                )
            ).lower()
            ==
            standard_name
        ):
            return name

    for name, var in ds.variables.items():

        units = str(
            var.attrs.get(
                "units",
                "",
            )
        ).lower()

        if (
            kind == "lat"
            and
            "degrees_north" in units
        ):
            return name

        if (
            kind == "lon"
            and
            "degrees_east" in units
        ):
            return name

    raise KeyError(
        f"Could not identify {kind} coordinate."
    )


def normalize_spatial_coordinates(
    ds,
    variable,
):
    lat_name = find_coordinate_name(
        ds,
        "lat",
    )

    lon_name = find_coordinate_name(
        ds,
        "lon",
    )

    rename = {}

    if lat_name != "lat":
        rename[lat_name] = "lat"

    if lon_name != "lon":
        rename[lon_name] = "lon"

    if rename:
        ds = ds.rename(
            rename
        )

    if (
        "lat" not in ds.coords
        and
        "lat" in ds.variables
    ):
        ds = ds.set_coords(
            "lat"
        )

    if (
        "lon" not in ds.coords
        and
        "lon" in ds.variables
    ):
        ds = ds.set_coords(
            "lon"
        )

    da = ds[
        variable
    ]

    core = {
        "time",
    }

    core.update(
        ds["lat"].dims
    )

    core.update(
        ds["lon"].dims
    )

    extra_dims = [
        d
        for d in da.dims
        if d not in core
    ]

    for d in extra_dims:

        if da.sizes[d] == 1:
            da = da.isel(
                {
                    d: 0,
                },
                drop=True,
            )

        else:
            raise ValueError(
                f"{variable} has unsupported extra dimension "
                f"{d} size={da.sizes[d]}"
            )

    return da


def subset_years_make_ym(
    da,
    start_year,
    end_year,
):
    if "time" not in da.dims:
        raise KeyError(
            "Expected time dimension."
        )

    year = da["time"].dt.year

    mask = (
        (year >= start_year)
        &
        (year <= end_year)
    )

    da = da.where(
        mask,
        drop=True,
    )

    if da.sizes.get(
        "time",
        0,
    ) == 0:
        raise ValueError(
            f"No data in {start_year}-{end_year}"
        )

    years = np.asarray(
        da["time"].dt.year.values,
        dtype=np.int32,
    )

    months = np.asarray(
        da["time"].dt.month.values,
        dtype=np.int16,
    )

    ym = (
        years * 100
        +
        months
    )

    # Remove duplicate YYYYMM timestamps.
    _, unique_idx = np.unique(
        ym,
        return_index=True,
    )

    unique_idx = np.sort(
        unique_idx
    )

    if len(unique_idx) < len(ym):

        da = da.isel(
            time=unique_idx
        )

        years = years[
            unique_idx
        ]

        months = months[
            unique_idx
        ]

        ym = ym[
            unique_idx
        ]

    da = da.assign_coords(
        ym=(
            "time",
            ym.astype(
                np.int32
            ),
        )
    )

    da = da.swap_dims(
        {
            "time": "ym",
        }
    )

    if "time" in da.coords:
        da = da.drop_vars(
            "time"
        )

    da = da.assign_coords(
        year=(
            "ym",
            years.astype(
                np.int16
            ),
        ),
        month=(
            "ym",
            months.astype(
                np.int8
            ),
        ),
    )

    return da.sortby(
        "ym"
    )


def normalize_regular_lon(
    da,
):
    lon = np.asarray(
        da["lon"].values,
        dtype=np.float64,
    )

    lon_new = (
        (lon + 180.0)
        %
        360.0
    ) - 180.0

    da = da.assign_coords(
        lon=lon_new
    )

    da = da.sortby(
        "lon"
    )

    lon_values = np.asarray(
        da["lon"].values
    )

    _, idx = np.unique(
        np.round(
            lon_values,
            10,
        ),
        return_index=True,
    )

    if len(idx) < len(
        lon_values
    ):
        da = da.isel(
            lon=np.sort(
                idx
            )
        )

    return da

def sphere_xyz(
    lat_deg,
    lon_deg,
):
    lat_rad = np.deg2rad(
        lat_deg
    )

    lon_rad = np.deg2rad(
        lon_deg
    )

    x = (
        np.cos(lat_rad)
        *
        np.cos(lon_rad)
    )

    y = (
        np.cos(lat_rad)
        *
        np.sin(lon_rad)
    )

    z = np.sin(
        lat_rad
    )

    return np.column_stack(
        [
            x,
            y,
            z,
        ]
    )


def regrid_curvilinear_nearest(
    da,
):
    lat = da[
        "lat"
    ]

    lon = da[
        "lon"
    ]

    if (
        lat.ndim != 2
        or
        lon.ndim != 2
    ):
        raise ValueError(
            "Curvilinear fallback requires 2-D lat/lon."
        )

    spatial_dims = lat.dims

    if lon.dims != spatial_dims:
        raise ValueError(
            "2-D lat/lon dimensions do not match."
        )

    lat_flat = np.asarray(
        lat.values,
        dtype=np.float64,
    ).ravel()

    lon_flat = np.asarray(
        lon.values,
        dtype=np.float64,
    ).ravel()

    finite = (
        np.isfinite(
            lat_flat
        )
        &
        np.isfinite(
            lon_flat
        )
    )

    source_indices = np.flatnonzero(
        finite
    )

    source_xyz = sphere_xyz(
        lat_flat[
            finite
        ],
        lon_flat[
            finite
        ],
    )

    target_lon_2d, target_lat_2d = np.meshgrid(
        TARGET_LON,
        TARGET_LAT,
    )

    target_xyz = sphere_xyz(
        target_lat_2d.ravel(),
        target_lon_2d.ravel(),
    )

    tree = cKDTree(
        source_xyz
    )

    _, nearest_local = tree.query(
        target_xyz,
        k=1,
    )

    nearest_flat = source_indices[
        nearest_local
    ]

    flat = da.stack(
        __src=spatial_dims
    )

    indexer = xr.DataArray(
        nearest_flat.astype(
            np.int64
        ),
        dims="__target",
    )

    selected = flat.isel(
        __src=indexer
    )

    selected = selected.transpose(
        "ym",
        "__target",
    )

    shape = (
        selected.sizes[
            "ym"
        ],
        len(TARGET_LAT),
        len(TARGET_LON),
    )

    data = selected.data.reshape(
        shape
    )

    out = xr.DataArray(
        data,
        dims=(
            "ym",
            "lat",
            "lon",
        ),
        coords={
            "ym": selected[
                "ym"
            ],
            "lat": TARGET_LAT,
            "lon": TARGET_LON,
        },
        name=da.name,
        attrs=dict(
            da.attrs
        ),
    )

    if "year" in da.coords:
        out = out.assign_coords(
            year=(
                "ym",
                np.asarray(
                    da["year"].values
                ),
            )
        )

    if "month" in da.coords:
        out = out.assign_coords(
            month=(
                "ym",
                np.asarray(
                    da["month"].values
                ),
            )
        )

    return out


def regrid_to_target(
    da,
):
    lat = da[
        "lat"
    ]

    lon = da[
        "lon"
    ]

    if (
        lat.ndim == 1
        and
        lon.ndim == 1
    ):
        da = normalize_regular_lon(
            da
        )

        if (
            da["lat"].values[0]
            >
            da["lat"].values[-1]
        ):
            da = da.sortby(
                "lat"
            )

        return da.interp(
            lat=TARGET_LAT,
            lon=TARGET_LON,
            method="linear",
        )

    if (
        lat.ndim == 2
        and
        lon.ndim == 2
    ):
        return regrid_curvilinear_nearest(
            da
        )

    raise ValueError(
        f"Unsupported grid: lat.ndim={lat.ndim}, "
        f"lon.ndim={lon.ndim}"
    )


def load_variable_existing(
    source_id,
    member_id,
    experiment_id,
    variable_id,
    start_year,
    end_year,
):
    paths = discover_existing_files(
        source_id,
        member_id,
        experiment_id,
        variable_id,
        start_year,
        end_year,
    )

    if not paths:
        raise FileNotFoundError(
            "No already-downloaded files found for "
            f"{source_id}/{member_id}/"
            f"{experiment_id}/{variable_id}"
        )

    print(
        f"    {variable_id} {experiment_id} "
        f"{start_year}-{end_year}: "
        f"{len(paths)} candidate file(s)"
    )

    ds = open_many(
        paths,
        variable_id,
    )

    try:
        da = normalize_spatial_coordinates(
            ds,
            variable_id,
        )

        da = subset_years_make_ym(
            da,
            start_year,
            end_year,
        )

        da = convert_water_storage_to_mm(
            da,
            variable_id,
        )

        da = regrid_to_target(
            da
        )

        da = da.where(
            COUNTRY_LAND_MASK
        )

        da = da.astype(
            "float32"
        ).load()

        return da

    finally:
        try:
            ds.close()
        except Exception:
            pass

        gc.collect()


# DIRECT STORAGE


def load_storage_existing(
    source_id,
    member_id,
    experiment_id,
    start_year,
    end_year,
):
    mrso = load_variable_existing(
        source_id,
        member_id,
        experiment_id,
        "mrso",
        start_year,
        end_year,
    )

    snw = load_variable_existing(
        source_id,
        member_id,
        experiment_id,
        "snw",
        start_year,
        end_year,
    )

    mrso, snw = xr.align(
        mrso,
        snw,
        join="inner",
    )

    if mrso.sizes.get(
        "ym",
        0,
    ) == 0:
        raise ValueError(
            "No common YYYYMM values between mrso and snw."
        )

    storage = (
        mrso
        +
        snw
    ).astype(
        "float32"
    )

    storage.name = "storage"

    storage.attrs.update(
        {
            "long_name":
                "Direct soil-snow water storage",
            "definition":
                "mrso + snw",
            "units":
                "mm",
        }
    )

    if "year" not in storage.coords:
        storage = storage.assign_coords(
            year=(
                "ym",
                (
                    storage[
                        "ym"
                    ].values
                    //
                    100
                ).astype(
                    np.int16
                ),
            )
        )

    if "month" not in storage.coords:
        storage = storage.assign_coords(
            month=(
                "ym",
                (
                    storage[
                        "ym"
                    ].values
                    %
                    100
                ).astype(
                    np.int8
                ),
            )
        )

    del mrso, snw
    gc.collect()

    return storage


def stitch_recent_storage(
    source_id,
    member_id,
    scenario,
):
    """
    2001-2023 =
        historical 2001-2014
        + scenario 2015-2023
    """

    hist = load_storage_existing(
        source_id,
        member_id,
        "historical",
        RECENT_START,
        HIST_RECENT_END,
    )

    scen = load_storage_existing(
        source_id,
        member_id,
        scenario,
        SCEN_RECENT_START,
        RECENT_END,
    )


    recent = xr.concat(
        [
            hist,
            scen,
        ],
        dim="ym",
        coords="minimal",
        compat="override",
        join="exact",
        combine_attrs="override",
    )

    recent = recent.sortby(
        "ym"
    )

    ym = np.asarray(
        recent[
            "ym"
        ].values
    )

    _, idx = np.unique(
        ym,
        return_index=True,
    )

    idx = np.sort(
        idx
    )

    if len(idx) < len(
        ym
    ):
        recent = recent.isel(
            ym=idx
        )

    years = (
        recent[
            "ym"
        ].values
        //
        100
    ).astype(
        np.int16
    )

    months = (
        recent[
            "ym"
        ].values
        %
        100
    ).astype(
        np.int8
    )

    recent = recent.assign_coords(
        year=(
            "ym",
            years,
        ),
        month=(
            "ym",
            months,
        ),
    )

    recent.attrs.update(
        {
            "period":
                "2001-2023",
            "construction":
                (
                    "historical 2001-2014 + "
                    f"{scenario} 2015-2023"
                ),
        }
    )

    del hist, scen
    gc.collect()

    return recent


def annual_mean_storage(
    storage,
):
    monthly_count = storage.groupby(
        "year"
    ).count(
        "ym"
    )

    annual = storage.groupby(
        "year"
    ).mean(
        "ym",
        skipna=True,
    )

    annual = annual.where(
        monthly_count
        >=
        MIN_MONTHS_PER_YEAR
    )

    annual.name = "annual_storage"

    return annual.astype(
        "float32"
    )


def build_fixed_reference(
    early_annual,
):
    reference_n = early_annual.count(
        "year"
    )

    reference_mean = early_annual.mean(
        "year",
        skipna=True,
    )

    reference_std = early_annual.std(
        "year",
        skipna=True,
        ddof=1,
    )

    good = (
        (
            reference_n
            >=
            MIN_REFERENCE_YEARS
        )
        &
        np.isfinite(
            reference_std
        )
        &
        (
            reference_std
            >=
            MIN_STD
        )
    )

    reference_mean = reference_mean.where(
        good
    )

    reference_std = reference_std.where(
        good
    )

    reference_n = reference_n.where(
        good,
        0,
    )

    reference_mean.name = (
        "reference_mean_1979_2000"
    )

    reference_std.name = (
        "reference_std_1979_2000"
    )

    reference_n.name = (
        "reference_n_1979_2000"
    )

    return (
        reference_mean,
        reference_std,
        reference_n,
    )


def standardize_annual(
    annual,
    reference_mean,
    reference_std,
):
    z = (
        (
            annual
            -
            reference_mean
        )
        /
        reference_std
    ).astype(
        "float32"
    )

    z.name = (
        "annual_standardized_storage"
    )

    z.attrs.update(
        {
            "definition":
                (
                    "(annual mrso+snw - "
                    "model 1979-2000 mean) / "
                    "model 1979-2000 std"
                ),
            "units":
                "1",
        }
    )

    return z


# TREND


def linear_trend_annual(
    annual_z,
    min_years,
):
    x = xr.DataArray(
        np.asarray(
            annual_z[
                "year"
            ].values,
            dtype=np.float64,
        ),
        dims="year",
        coords={
            "year":
                annual_z[
                    "year"
                ],
        },
    )

    valid = np.isfinite(
        annual_z
    )

    n = valid.sum(
        "year"
    ).astype(
        "int16"
    )

    x_masked = x.where(
        valid
    )

    y_masked = annual_z.where(
        valid
    )

    xbar = (
        x_masked.sum(
            "year",
            skipna=True,
        )
        /
        n
    )

    ybar = (
        y_masked.sum(
            "year",
            skipna=True,
        )
        /
        n
    )

    dx = (
        x
        -
        xbar
    )

    dy = (
        annual_z
        -
        ybar
    )

    sxx = (
        (
            dx.where(
                valid
            )
            **
            2
        )
        .sum(
            "year",
            skipna=True,
        )
    )

    sxy = (
        (
            dx.where(
                valid
            )
            *
            dy.where(
                valid
            )
        )
        .sum(
            "year",
            skipna=True,
        )
    )

    slope = (
        sxy
        /
        sxx
    )

    intercept = (
        ybar
        -
        slope
        *
        xbar
    )

    fitted = (
        slope
        *
        x
        +
        intercept
    )

    residual = (
        annual_z
        -
        fitted
    ).where(
        valid
    )

    sse = (
        residual
        **
        2
    ).sum(
        "year",
        skipna=True,
    )

    df = (
        n.astype(
            "float64"
        )
        -
        2.0
    )

    mse = (
        sse
        /
        df
    )

    se_slope = np.sqrt(
        mse
        /
        sxx
    )

    t_stat = (
        slope
        /
        se_slope
    )

    p_values = (
        2.0
        *
        special.stdtr(
            np.asarray(
                df.values,
                dtype=np.float64,
            ),
            -np.abs(
                np.asarray(
                    t_stat.values,
                    dtype=np.float64,
                )
            ),
        )
    )

    p = xr.DataArray(
        p_values,
        dims=slope.dims,
        coords=slope.coords,
        name="p_value",
    )

    good = (
        (
            n
            >=
            min_years
        )
        &
        np.isfinite(
            slope
        )
        &
        np.isfinite(
            p
        )
        &
        (
            sxx
            >
            0
        )
    )

    slope = slope.where(
        good
    ).astype(
        "float32"
    )

    p = p.where(
        good
    ).astype(
        "float32"
    )

    n = n.where(
        good,
        0,
    ).astype(
        "int16"
    )

    slope.name = "slope"
    p.name = "p_value"
    n.name = "n_years"

    slope.attrs[
        "units"
    ] = (
        "standardized_storage per year"
    )

    return (
        slope,
        p,
        n,
    )

# DDWW TRANSITION CLASSIFICATION


def classify_transition(
    slope_first,
    slope_second,
):
    first_pos = (
        slope_first
        >
        SLOPE_EPS
    )

    first_neg = (
        slope_first
        <
        -SLOPE_EPS
    )

    second_pos = (
        slope_second
        >
        SLOPE_EPS
    )

    second_neg = (
        slope_second
        <
        -SLOPE_EPS
    )

    valid = (
        np.isfinite(
            slope_first
        )
        &
        np.isfinite(
            slope_second
        )
    )

    cls = xr.zeros_like(
        slope_first,
        dtype=np.int8,
    )

    cls = xr.where(
        valid
        &
        first_pos
        &
        second_pos,
        1,
        cls,
    )

    cls = xr.where(
        valid
        &
        first_neg
        &
        second_neg,
        2,
        cls,
    )

    cls = xr.where(
        valid
        &
        first_pos
        &
        second_neg,
        3,
        cls,
    )

    cls = xr.where(
        valid
        &
        first_neg
        &
        second_pos,
        4,
        cls,
    )

    cls = cls.where(
        valid,
        0,
    ).astype(
        "int8"
    )

    cls.name = "class_code"

    cls.attrs.update(
        {
            "flag_values":
                "0,1,2,3,4",
            "flag_meanings":
                "unclassified WW DD WD DW",
            "WW":
                "positive first-period trend; positive second-period trend",
            "DD":
                "negative first-period trend; negative second-period trend",
            "WD":
                "positive first-period trend; negative second-period trend",
            "DW":
                "negative first-period trend; positive second-period trend",
        }
    )

    return cls


def significance_intensity(
    p_first,
    p_second,
):
    valid = (
        np.isfinite(
            p_first
        )
        &
        np.isfinite(
            p_second
        )
    )

    intensity = (
        (
            p_first
            <
            P_THRESHOLD
        ).astype(
            np.int8
        )
        +
        (
            p_second
            <
            P_THRESHOLD
        ).astype(
            np.int8
        )
    )

    intensity = intensity.where(
        valid,
        -1,
    ).astype(
        "int8"
    )

    intensity.name = (
        "significance_intensity"
    )

    intensity.attrs.update(
        {
            "description":
                (
                    "0 neither significant; "
                    "1 one significant; "
                    "2 both significant; "
                    "-1 unavailable"
                ),
            "p_threshold":
                P_THRESHOLD,
        }
    )

    return intensity



def model_output_path(
    source_id,
    member_id,
    scenario,
    future_period_name,
):
    return DIR_MODEL / (
        f"{safe_name(source_id)}_"
        f"{safe_name(member_id)}_"
        f"{scenario}_"
        f"{future_period_name}_"
        f"manuscript_consistent.nc"
    )



def process_model_case(
    source_id,
    member_id,
    scenario,
    future_period_name,
    future_start,
    future_end,
):
    out_path = model_output_path(
        source_id,
        member_id,
        scenario,
        future_period_name,
    )

    if (
        out_path.exists()
        and
        not OVERWRITE
    ):
        print(
            f"  Existing model result: {out_path.name}"
        )

        return out_path

    print_header(
        f"MODEL CASE: {source_id} / {member_id} / "
        f"{scenario} / {future_period_name}"
    )



    early_storage = load_storage_existing(
        source_id,
        member_id,
        "historical",
        EARLY_START,
        EARLY_END,
    )

    early_annual = annual_mean_storage(
        early_storage
    )

    (
        reference_mean,
        reference_std,
        reference_n,
    ) = build_fixed_reference(
        early_annual
    )

    early_z = standardize_annual(
        early_annual,
        reference_mean,
        reference_std,
    )

    (
        early_slope,
        early_p,
        early_n,
    ) = linear_trend_annual(
        early_z,
        MIN_EARLY_TREND_YEARS,
    )


    recent_storage = stitch_recent_storage(
        source_id,
        member_id,
        scenario,
    )

    recent_annual = annual_mean_storage(
        recent_storage
    )

    recent_z = standardize_annual(
        recent_annual,
        reference_mean,
        reference_std,
    )

    (
        recent_slope,
        recent_p,
        recent_n,
    ) = linear_trend_annual(
        recent_z,
        MIN_RECENT_TREND_YEARS,
    )


    future_storage = load_storage_existing(
        source_id,
        member_id,
        scenario,
        future_start,
        future_end,
    )

    future_annual = annual_mean_storage(
        future_storage
    )

    future_z = standardize_annual(
        future_annual,
        reference_mean,
        reference_std,
    )

    (
        future_slope,
        future_p,
        future_n,
    ) = linear_trend_annual(
        future_z,
        MIN_FUTURE_TREND_YEARS,
    )


    historical_class = classify_transition(
        early_slope,
        recent_slope,
    )

    historical_sig = significance_intensity(
        early_p,
        recent_p,
    )

    future_class = classify_transition(
        recent_slope,
        future_slope,
    )

    future_sig = significance_intensity(
        recent_p,
        future_p,
    )


    ds_out = xr.Dataset(
        {
            "reference_mean_1979_2000":
                reference_mean.astype(
                    "float32"
                ),

            "reference_std_1979_2000":
                reference_std.astype(
                    "float32"
                ),

            "reference_n_1979_2000":
                reference_n.astype(
                    "int16"
                ),

            "early_annual_z":
                early_z.astype(
                    "float32"
                ),

            "recent_annual_z":
                recent_z.astype(
                    "float32"
                ),

            "future_annual_z":
                future_z.astype(
                    "float32"
                ),

            "early_slope":
                early_slope.astype(
                    "float32"
                ),

            "early_p":
                early_p.astype(
                    "float32"
                ),

            "early_n_years":
                early_n.astype(
                    "int16"
                ),

            "recent_slope":
                recent_slope.astype(
                    "float32"
                ),

            "recent_p":
                recent_p.astype(
                    "float32"
                ),

            "recent_n_years":
                recent_n.astype(
                    "int16"
                ),

            "future_slope":
                future_slope.astype(
                    "float32"
                ),

            "future_p":
                future_p.astype(
                    "float32"
                ),

            "future_n_years":
                future_n.astype(
                    "int16"
                ),

            "historical_transition_class":
                historical_class.astype(
                    "int8"
                ),

            "historical_significance_intensity":
                historical_sig.astype(
                    "int8"
                ),

            "future_transition_class":
                future_class.astype(
                    "int8"
                ),

            "future_significance_intensity":
                future_sig.astype(
                    "int8"
                ),

            "country_land_mask":
                COUNTRY_LAND_MASK.astype(
                    "int8"
                ),
        }
    )

    ds_out.attrs.update(
        {
            "source_id":
                source_id,

            "member_id":
                member_id,

            "scenario":
                scenario,

            "storage_definition":
                "S = mrso + snw",

            "early_period":
                "1979-2000",

            "recent_period":
                "2001-2023",

            "recent_period_construction":
                (
                    "historical 2001-2014 + "
                    f"{scenario} 2015-2023"
                ),

            "future_period":
                f"{future_start}-{future_end}",

            "future_period_name":
                future_period_name,

            "normalization_reference":
                (
                    "model-specific annual direct-storage "
                    "mean and std over 1979-2000"
                ),

            "historical_transition":
                "trend(1979-2000) -> trend(2001-2023)",

            "future_transition":
                (
                    "trend(2001-2023) -> "
                    f"trend({future_start}-{future_end})"
                ),

            "classification_logic":
                (
                    "WW=+/+, DD=-/-, WD=+/-, DW=-/+"
                ),

            "p_threshold":
                P_THRESHOLD,

            "target_grid_resolution_degrees":
                TARGET_RESOLUTION_DEG,

            "land_mask_source":
                str(WORLD_SHP),
        }
    )

    write_netcdf(
        ds_out,
        out_path,
    )

    print(
        f"\nSaved model result:\n{out_path}"
    )

    del (
        early_storage,
        early_annual,
        early_z,
        recent_storage,
        recent_annual,
        recent_z,
        future_storage,
        future_annual,
        future_z,
        ds_out,
    )

    gc.collect()

    return out_path




AUDIT_FILE = (ROOT / "DIRECT_STORAGE_FUTURE_ANALYSIS_CONSISTENT"
              / "06_FIG4_ICEFREE_AREA_AUDIT"
              / "REBUILD_CASES.csv")
if not AUDIT_FILE.is_file():
    raise FileNotFoundError(f"Run revised ensemble analysis FIRST: {AUDIT_FILE}")
rebuild_table = pd.read_csv(AUDIT_FILE)
required_case_columns = {"source_id", "member_id", "scenario", "future_period", "reason"}
if not required_case_columns.issubset(rebuild_table.columns):
    raise RuntimeError("Malformed targeted rebuild CSV")
rebuild_table = rebuild_table.loc[
    rebuild_table["reason"].astype(str).eq("missing_months_from_source_filenames")
].drop_duplicates(subset=["source_id", "member_id", "scenario", "future_period"])

MAX_STRICT_CASES_PER_RUN = 12
if len(rebuild_table) > MAX_STRICT_CASES_PER_RUN:
    warnings.warn(f"Found {len(rebuild_table)} targeted cases, processing first "
                  f"{MAX_STRICT_CASES_PER_RUN} only; edit audit CSV and rerun for more")
    rebuild_table = rebuild_table.head(MAX_STRICT_CASES_PER_RUN)
SELECTED_CASES = set(map(tuple, rebuild_table[
    ["source_id", "member_id", "scenario", "future_period"]
].itertuples(index=False, name=None)))
if not SELECTED_CASES:
    print("No known filename-month gaps.")
    print("full per-pixel month validation is still not proven.")
    raise SystemExit(0)
else:
    available_pairs = set(zip(selected_pairs.source_id.astype(str),
                              selected_pairs.member_id.astype(str)))
    absent = [(m, member) for m, member, sc, per in SELECTED_CASES
              if (str(m), str(member)) not in available_pairs]
    if absent:
        raise RuntimeError(f"Selected target models are not present in the current manifest selection: {absent}")
    print(f"Strict-12-month reprocessing of {len(SELECTED_CASES)} selected cases only.")

processing_rows = []

for model_index, row in enumerate(
    selected_pairs.itertuples(
        index=False
    ),
    start=1,
):
    source_id = row.source_id
    member_id = row.member_id

    print_header(
        f"MODEL {model_index}/{len(selected_pairs)}: "
        f"{source_id} / {member_id}"
    )

    for scenario, windows in FUTURE_WINDOWS.items():

        for future_period_name, (
            future_start,
            future_end,
        ) in windows.items():

            if (source_id, member_id, scenario, future_period_name) not in SELECTED_CASES:
                continue

            try:
                out_path = process_model_case(
                    source_id,
                    member_id,
                    scenario,
                    future_period_name,
                    future_start,
                    future_end,
                )

                processing_rows.append(
                    {
                        "source_id":
                            source_id,

                        "member_id":
                            member_id,

                        "scenario":
                            scenario,

                        "future_period":
                            future_period_name,

                        "status":
                            "success",

                        "output_path":
                            str(out_path),

                        "error":
                            "",
                    }
                )

            except Exception as exc:

                print(
                    "\nMODEL CASE FAILED:"
                )

                print(
                    exc
                )

                processing_rows.append(
                    {
                        "source_id":
                            source_id,

                        "member_id":
                            member_id,

                        "scenario":
                            scenario,

                        "future_period":
                            future_period_name,

                        "status":
                            "failed",

                        "output_path":
                            "",

                        "error":
                            repr(
                                exc
                            ),

                        "traceback":
                            traceback.format_exc(),
                    }
                )

            pd.DataFrame(
                processing_rows
            ).to_csv(
                DIR_LOGS
                / "model_processing_log.csv",
                index=False,
            )

            gc.collect()


